# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.2 — Baseline Modeling

### Objective

Build the first baseline machine learning models for customer churn prediction.

The goals of this phase are:

- Create a leakage-safe preprocessing pipeline
- Train baseline Logistic Regression models
- Compare Model A and Model B
- Evaluate churn prediction performance
- Investigate potential leakage effects from retention-related features

---

### Modeling Variants

#### Model A
Includes retention-related features.

#### Model B
Excludes retention-related features.

---

### Evaluation Metrics

Primary Metrics:

- ROC-AUC
- Recall
- F1 Score

Secondary Metrics:

- Precision
- Accuracy

---

### Dataset Source

Prepared during Phase 6.1.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1: Load Dataset

Load the feature-engineered dataset and recreate the modeling datasets developed during Phase 6.1.

This ensures the modeling notebook is self-contained and reproducible.

In [2]:
# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("../data/processed/feature_engineered.csv")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

Dataset Loaded Successfully
Shape: (51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_18056\2978817408.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/feature_engineered.csv")


In [3]:
df.head()

,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,UnansweredCalls,CustomerCareCalls,ThreewayCalls,ReceivedCalls,OutboundCalls,InboundCalls,PeakCallsInOut,OffPeakCallsInOut,DroppedBlockedCalls,CallForwardingCalls,CallWaitingCalls,MonthsInService,UniqueSubs,ActiveSubs,ServiceArea,Handsets,HandsetModels,CurrentEquipmentDays,AgeHH1,AgeHH2,ChildrenInHH,HandsetRefurbished,HandsetWebCapable,TruckOwner,RVOwner,Homeownership,BuysViaMailOrder,RespondsToMailOffers,OptOutMailings,NonUSTravel,OwnsComputer,HasCreditCard,RetentionCalls,RetentionOffersAccepted,NewCellphoneUser,NotNewCellphoneUser,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus,AgeHH2_Missing,HandsetPrice_Missing,RevenueGroup_Missing,CustomerLifecycleStage,CustomerValueProxy,CreditRating_Encoded,RegionCode,MarketCode,AreaCode
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,0.7,0.7,6.3,0.0,0.0,97.2,0.0,0.0,58.0,24.0,1.3,0.0,0.3,61,2,1,SEAPOR503,2.0,2.0,361.0,62.0,44.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,Yes,Yes,1,0,No,No,0,4,No,0,30.0,Yes,1-Highest,Suburban,Professional,No,True,0,0,Loyal,1464.00,7,SEA,POR,503
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,0.3,0.0,2.7,0.0,0.0,0.0,0.0,0.0,5.0,1.0,0.3,0.0,0.0,58,1,1,PITHOM412,2.0,1.0,1504.0,40.0,42.0,Yes,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,Yes,No,0,5,No,0,30.0,No,4-Medium,Suburban,Professional,Yes,False,0,0,Loyal,985.42,4,PIT,HOM,412
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.3,0.0,1.3,3.7,0.0,0.0,0.0,60,1,1,MILMIL414,1.0,1.0,1812.0,26.0,26.0,Yes,No,No,No,No,Unknown,No,No,No,No,No,Yes,0,0,Yes,No,0,6,No,0,60.0,No,3-Good,Town,Crafts,Yes,False,1,0,Loyal,2280.00,5,MIL,MIL,414
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,52.0,7.7,76.0,4.3,1.3,200.3,370.3,147.0,555.7,303.7,59.7,0.0,22.7,59,2,2,PITHOM412,9.0,4.0,458.0,30.0,44.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,No,Yes,0,0,Yes,No,0,6,No,0,10.0,No,4-Medium,Other,Other,No,True,0,0,Loyal,4854.52,4,PIT,HOM,412
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,53,2,2,OKCTUL918,4.0,3.0,852.0,46.0,54.0,No,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,No,Yes,0,9,No,1,10.0,No,1-Highest,Other,Professional,Yes,False,0,0,Loyal,908.42,7,OKC,TUL,918


## Step 2: Recreate Modeling Datasets

Before training baseline models, recreate the final modeling datasets prepared during Phase 6.1.

The following decisions are applied:

- Encode target variable
- Remove CustomerID
- Remove ServiceArea
- Remove MarketCode
- Remove AreaCode
- Remove original CreditRating
- Create Model A and Model B datasets

In [4]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Target encoding completed.")

Target encoding completed.


In [5]:
# ==========================================
# COLUMN REMOVALS
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating"
]

model_df = df.drop(columns=columns_to_drop)

print("Columns removed successfully.")
print("Shape:", model_df.shape)

Columns removed successfully.
Shape: (51047, 62)


## Step 3: Create Model A and Model B

Two modeling datasets are created.

### Model A
Includes retention-related variables.

### Model B
Excludes retention-related variables.

The performance difference will later be used to evaluate potential leakage effects.

In [6]:
# ==========================================
# RETENTION FEATURES
# ==========================================

retention_features = [
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

In [7]:
# ==========================================
# MODEL A
# ==========================================

model_a_df = model_df.copy()

print("Model A Shape:")
print(model_a_df.shape)

Model A Shape:
(51047, 62)


In [8]:
# ==========================================
# MODEL B
# ==========================================

model_b_df = model_df.drop(
    columns=retention_features
)

print("Model B Shape:")
print(model_b_df.shape)

Model B Shape:
(51047, 59)


In [9]:
# ==========================================
# FEATURES & TARGET
# ==========================================

X_a = model_a_df.drop(columns=["Churn"])
y_a = model_a_df["Churn"]

X_b = model_b_df.drop(columns=["Churn"])
y_b = model_b_df["Churn"]

print("Model A Features:", X_a.shape)
print("Model B Features:", X_b.shape)

Model A Features: (51047, 61)
Model B Features: (51047, 58)


In [10]:
# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train_a, X_valid_a, y_train_a, y_valid_a = train_test_split(
    X_a,
    y_a,
    test_size=0.20,
    stratify=y_a,
    random_state=42
)

X_train_b, X_valid_b, y_train_b, y_valid_b = train_test_split(
    X_b,
    y_b,
    test_size=0.20,
    stratify=y_b,
    random_state=42
)

print("Train-validation split completed.")

Train-validation split completed.


In [11]:
print("Model A Train:", X_train_a.shape)
print("Model A Valid:", X_valid_a.shape)

print()

print("Model B Train:", X_train_b.shape)
print("Model B Valid:", X_valid_b.shape)

Model A Train: (40837, 61)
Model A Valid: (10210, 61)

Model B Train: (40837, 58)
Model B Valid: (10210, 58)


In [12]:
numerical_features_a = X_train_a.select_dtypes(
    exclude=["object"]
).columns.tolist()

In [13]:
categorical_features_a = X_train_a.select_dtypes(
    include=["object"]
).columns.tolist()

In [14]:
OneHotEncoder(
    handle_unknown="ignore"
)

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


## Step 4: Feature Type Identification

Machine learning preprocessing depends on feature type.

Features are divided into:

### Numerical Features

These features can be passed directly to the model.

### Categorical Features

These features require encoding before model training.

The feature inventory will be used to build a leakage-safe preprocessing pipeline.

In [15]:
# ==========================================
# MODEL A FEATURE TYPES
# ==========================================

numerical_features_a = X_train_a.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features_a = X_train_a.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features :", len(numerical_features_a))
print("Categorical Features :", len(categorical_features_a))

Numerical Features : 40
Categorical Features : 21


In [16]:
# Categorical Features

categorical_features_a

['ChildrenInHH',
 'HandsetRefurbished',
 'HandsetWebCapable',
 'TruckOwner',
 'RVOwner',
 'Homeownership',
 'BuysViaMailOrder',
 'RespondsToMailOffers',
 'OptOutMailings',
 'NonUSTravel',
 'OwnsComputer',
 'HasCreditCard',
 'NewCellphoneUser',
 'NotNewCellphoneUser',
 'OwnsMotorcycle',
 'MadeCallToRetentionTeam',
 'PrizmCode',
 'Occupation',
 'MaritalStatus',
 'CustomerLifecycleStage',
 'RegionCode']

In [17]:
# Numerical Features (first 20)

numerical_features_a[:20]

['MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'DroppedCalls',
 'BlockedCalls',
 'UnansweredCalls',
 'CustomerCareCalls',
 'ThreewayCalls',
 'ReceivedCalls',
 'OutboundCalls',
 'InboundCalls',
 'PeakCallsInOut',
 'OffPeakCallsInOut',
 'DroppedBlockedCalls',
 'CallForwardingCalls']

## Step 5: Create Preprocessing Pipeline

Categorical variables cannot be directly consumed by Logistic Regression.

Therefore:

- Numerical variables will pass through unchanged.
- Categorical variables will be One-Hot Encoded.

A ColumnTransformer ensures that preprocessing is applied consistently and without data leakage.

In [18]:
# ==========================================
# PREPROCESSOR IMPORTS
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [19]:
# ==========================================
# COLUMN TRANSFORMER
# ==========================================

preprocessor_a = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_a
        )
    ],
    remainder="passthrough"
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [20]:
type(preprocessor_a)

sklearn.compose._column_transformer.ColumnTransformer

## Step 6: Build Baseline Logistic Regression Pipeline

A machine learning pipeline combines:

1. Preprocessing
2. Model Training

Benefits:

- Prevents data leakage
- Ensures reproducibility
- Simplifies deployment
- Follows industry best practices

The first baseline model will use Logistic Regression.

In [21]:
# ==========================================
# LOGISTIC REGRESSION IMPORT
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [22]:
# ==========================================
# MODEL A PIPELINE
# ==========================================

pipeline_a = Pipeline(
    steps=[
        ("preprocessor", preprocessor_a),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Model A pipeline created.")

Model A pipeline created.


In [23]:
type(pipeline_a)

sklearn.pipeline.Pipeline

In [24]:
pipeline_a

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Step 7: Train Baseline Model A

Train the Logistic Regression baseline model using:

- Retention features included
- OneHot Encoding pipeline
- Stratified training dataset

This serves as the first benchmark model.

In [25]:
# ==========================================
# TRAIN MODEL A
# ==========================================

pipeline_a.fit(
    X_train_a,
    y_train_a
)

print("Model A training completed.")

Model A training completed.


C:\Users\Akshit\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
# ==========================================
# MODEL A PREDICTIONS
# ==========================================

y_pred_a = pipeline_a.predict(X_valid_a)

y_prob_a = pipeline_a.predict_proba(X_valid_a)[:, 1]

print("Predictions generated.")

Predictions generated.


In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [28]:
# ==========================================
# MODEL A METRICS
# ==========================================

accuracy_a = accuracy_score(
    y_valid_a,
    y_pred_a
)

precision_a = precision_score(
    y_valid_a,
    y_pred_a
)

recall_a = recall_score(
    y_valid_a,
    y_pred_a
)

f1_a = f1_score(
    y_valid_a,
    y_pred_a
)

roc_auc_a = roc_auc_score(
    y_valid_a,
    y_prob_a
)

print("Accuracy :", round(accuracy_a, 4))
print("Precision:", round(precision_a, 4))
print("Recall   :", round(recall_a, 4))
print("F1 Score :", round(f1_a, 4))
print("ROC AUC  :", round(roc_auc_a, 4))

Accuracy : 0.7118
Precision: 0.4973
Recall   : 0.0313
F1 Score : 0.0588
ROC AUC  : 0.6083


In [29]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_valid_a,
    y_pred_a
)

cm

array([[7175,   93],
       [2850,   92]])

In [30]:
pd.DataFrame(
    cm,
    index=["Actual_No", "Actual_Yes"],
    columns=["Pred_No", "Pred_Yes"]
)

,Pred_No,Pred_Yes
Actual_No,7175,93
Actual_Yes,2850,92


## Experiment 2: Class-Balanced Logistic Regression

The baseline model showed extremely low recall.

To address class imbalance, class weights are introduced.

This increases the penalty for misclassifying churners and encourages the model to identify more positive cases.

In [31]:
# ==========================================
# BALANCED LOGISTIC REGRESSION
# ==========================================

pipeline_a_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor_a),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Balanced pipeline created.")

Balanced pipeline created.


In [32]:
# Train balanced model

pipeline_a_balanced.fit(
    X_train_a,
    y_train_a
)

print("Balanced model trained.")

Balanced model trained.


C:\Users\Akshit\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [33]:
# Predictions

y_pred_balanced = pipeline_a_balanced.predict(
    X_valid_a
)

y_prob_balanced = pipeline_a_balanced.predict_proba(
    X_valid_a
)[:, 1]

In [34]:
# Metrics

print("Accuracy :",
      round(accuracy_score(y_valid_a, y_pred_balanced), 4))

print("Precision:",
      round(precision_score(y_valid_a, y_pred_balanced), 4))

print("Recall   :",
      round(recall_score(y_valid_a, y_pred_balanced), 4))

print("F1 Score :",
      round(f1_score(y_valid_a, y_pred_balanced), 4))

print("ROC AUC  :",
      round(roc_auc_score(y_valid_a, y_prob_balanced), 4))

Accuracy : 0.5752
Precision: 0.3552
Recall   : 0.5816
F1 Score : 0.441
ROC AUC  : 0.6085


## Experiment 3: Model B (Without Retention Features)

To evaluate the impact of retention-related variables, a second balanced Logistic Regression model is trained.

Differences from Model A:

- RetentionCalls removed
- RetentionOffersAccepted removed
- MadeCallToRetentionTeam removed

All other preprocessing and evaluation steps remain identical.

This allows a fair comparison between the two models.

In [35]:
# ==========================================
# MODEL B FEATURE TYPES
# ==========================================

numerical_features_b = X_train_b.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features_b = X_train_b.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features :", len(numerical_features_b))
print("Categorical Features :", len(categorical_features_b))

Numerical Features : 38
Categorical Features : 20


In [36]:
# ==========================================
# MODEL B PREPROCESSOR
# ==========================================

preprocessor_b = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_b
        )
    ],
    remainder="passthrough"
)

print("Model B preprocessor created.")

Model B preprocessor created.


In [37]:
# ==========================================
# MODEL B BALANCED PIPELINE
# ==========================================

pipeline_b_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor_b),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Model B pipeline created.")

Model B pipeline created.


In [38]:
# ==========================================
# TRAIN MODEL B
# ==========================================

pipeline_b_balanced.fit(
    X_train_b,
    y_train_b
)

print("Model B training completed.")

Model B training completed.


C:\Users\Akshit\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [39]:
# ==========================================
# PREDICTIONS
# ==========================================

y_pred_b = pipeline_b_balanced.predict(
    X_valid_b
)

y_prob_b = pipeline_b_balanced.predict_proba(
    X_valid_b
)[:, 1]

print("Predictions generated.")

Predictions generated.


In [40]:
# ==========================================
# MODEL B METRICS
# ==========================================

accuracy_b = accuracy_score(
    y_valid_b,
    y_pred_b
)

precision_b = precision_score(
    y_valid_b,
    y_pred_b
)

recall_b = recall_score(
    y_valid_b,
    y_pred_b
)

f1_b = f1_score(
    y_valid_b,
    y_pred_b
)

roc_auc_b = roc_auc_score(
    y_valid_b,
    y_prob_b
)

print("Accuracy :", round(accuracy_b, 4))
print("Precision:", round(precision_b, 4))
print("Recall   :", round(recall_b, 4))
print("F1 Score :", round(f1_b, 4))
print("ROC AUC  :", round(roc_auc_b, 4))

Accuracy : 0.5697
Precision: 0.3533
Recall   : 0.5942
F1 Score : 0.4431
ROC AUC  : 0.603
